# gpu-optical-photon-toy on Google Colab

This notebook builds the project with CUDA on a Colab GPU and runs:

1. the full test suite (CPU unit/validation tests **and** GPU tests);
2. a second "reproducibility" build (portable math + `--fmad=false`), whose GPU test asserts that CPU and GPU histories are **bitwise identical**;
3. the analytic validation scans on the **GPU backend**;
4. the CPU vs GPU comparison (photon-by-photon with the same seed, statistical tests with independent seeds);
5. the throughput benchmarks;

and writes all figures to `docs/figures/` plus a zip you can download.

**Before running:** *Runtime → Change runtime type → GPU* (a T4 is fine). The whole notebook takes roughly 10–20 minutes on a T4.

In [ ]:
!nvidia-smi
!nvcc --version
!cmake --version | head -1

## 1. Get the code

Set `REPO_URL` to the GitHub repository. If the repository is private or not yet published, upload a zip of the repository instead (left sidebar → Files → upload) and set `ZIP_NAME`.

In [ ]:
REPO_URL = "https://github.com/YOUR_GITHUB_USER/gpu-optical-photon-toy.git"  # <- edit
ZIP_NAME = ""  # e.g. "gpu-optical-photon-toy.zip" if you uploaded a zip instead

import os, subprocess
os.chdir("/content")
if ZIP_NAME:
    !rm -rf repo && mkdir repo && unzip -q "$ZIP_NAME" -d repo
    # use the single top-level folder inside the zip if there is one
    inner = [d for d in os.listdir("repo") if os.path.isdir(os.path.join("repo", d))]
    root = os.path.join("/content/repo", inner[0]) if len(inner) == 1 else "/content/repo"
else:
    !rm -rf repo && git clone --depth 1 "$REPO_URL" repo
    root = "/content/repo"
os.chdir(root)
print("working in", os.getcwd())
!git log --oneline -1 2>/dev/null || true

## 2. Build with CUDA and run all tests

`CMAKE_CUDA_ARCHITECTURES` is taken from the GPU actually present (T4 = 75, L4 = 89, A100 = 80).

In [ ]:
cc = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0]
ARCH = cc.replace(".", "")
print("CUDA architecture:", ARCH)
!cmake -S . -B build-gpu -DCMAKE_BUILD_TYPE=Release -DWITH_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$ARCH > /dev/null
!cmake --build build-gpu -j 4 2>&1 | grep -E "error|warning|Built target" | tail -20

In [ ]:
!cd build-gpu && ctest --output-on-failure 2>&1 | tail -25

The GPU tests print their key numbers (pulls, agreement fractions):

In [ ]:
!./build-gpu/tests/gpu_tests 2>&1 | grep -E "^\[gpu\]|passed|failed"

## 3. Reproducibility build: portable math + no FMA contraction

With `OPTPHOT_PORTABLE_MATH=ON` the code uses its own `log`/`sincos`/`cbrt` built only from correctly rounded IEEE operations, and `OPTPHOT_CUDA_FMAD=OFF` stops nvcc from fusing `a*b+c` into FMA. The GPU test then **requires** every photon record to be bitwise identical to the CPU one.

In [ ]:
!cmake -S . -B build-repro -DCMAKE_BUILD_TYPE=Release -DWITH_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$ARCH -DOPTPHOT_PORTABLE_MATH=ON -DOPTPHOT_CUDA_FMAD=OFF > /dev/null
!cmake --build build-repro -j 4 2>&1 | grep -E "error|warning" | tail -20
!./build-repro/tests/gpu_tests 2>&1 | grep -E "^\[gpu\]|passed|failed|FAILED"

## 4. Analytic validation on the GPU backend

Same scans as the CPU figures in `docs/figures/validation_*.png`, run with `--backend gpu`; figures are written as `docs/figures/gpu_validation_*.png`.

In [ ]:
!pip -q install numpy scipy matplotlib
!python python/run_validation.py --exe build-gpu/apps/optphot --backend gpu --n 2e6 --prefix gpu_ --summary results/validation_summary_gpu.json

## 5. CPU vs GPU comparison

For each setup `optphot_compare` runs the CPU (seed s), the GPU (seed s: photon-by-photon comparison) and the GPU again with seed s+1 (independent sample: statistical tests). The last setup uses the reproducibility build.

In [ ]:
N = "2e6"
setups = {
    "black_box": "configs/solid_angle.cfg",
    "absorption": "configs/absorption.cfg",
    "polished": "configs/default.cfg",
    "lambertian": "configs/wrapped_lambertian.cfg",
}
args = []
for label, cfg in setups.items():
    !./build-gpu/apps/optphot_compare --config $cfg --n_photons $N --out results/compare/$label | tail -12
    args.append(f"{label}=results/compare/{label}")
!./build-repro/apps/optphot_compare --config configs/wrapped_lambertian.cfg --n_photons $N --out results/compare/lambertian_repro | tail -12
args.append("lambertian_repro=results/compare/lambertian_repro")
!python python/plot_compare.py {" ".join(args)}

## 6. Benchmarks

Throughput vs number of photons. Colab VMs have only 2 vCPUs, so the CPU numbers here are **not** representative of a workstation; the local CPU benchmark in the README was measured separately. Records mode is limited to 10^7 photons (29 bytes per photon on the host).

In [ ]:
!./build-gpu/apps/optphot_bench --config configs/default.cfg --sizes 1e4,1e5,1e6,1e7,1e8 --repeat 3 --max_cpu1 1e6 --csv results/bench_colab_default.csv | tail -5
!./build-gpu/apps/optphot_bench --config configs/wrapped_lambertian.cfg --sizes 1e4,1e5,1e6,1e7,1e8 --repeat 3 --max_cpu1 1e6 --csv results/bench_colab_lambertian.csv | tail -5
!python python/plot_benchmarks.py results/bench_colab_default.csv --label "default cube (polished walls, centre source)" --prefix benchmark_colab_default
!python python/plot_benchmarks.py results/bench_colab_lambertian.csv --label "wrapped Lambertian cube, volume source" --prefix benchmark_colab_lambertian

## 7. (Optional) Profiling

Colab images usually ship Nsight Systems (`nsys`) and Nsight Compute (`ncu`) with the CUDA toolkit; if they are missing, these cells just report it. See `docs/PROFILING.md` for what to look at.

In [ ]:
!which nsys ncu || echo "Nsight tools not found in this image"
!nsys profile -o results/optphot_tally --force-overwrite true ./build-gpu/apps/optphot --backend gpu --mode tally --n_photons 1e7 > /dev/null 2>&1 && nsys stats --report cuda_gpu_kern_sum results/optphot_tally.nsys-rep | tail -8 || true
!ncu --kernel-name regex:transport_records --launch-count 1 --section SpeedOfLight --section Occupancy --section WarpStateStats ./build-gpu/apps/optphot --backend gpu --n_photons 1e6 2>&1 | tail -60 || true

## 8. Look at the figures and download everything

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("docs/figures/gpu_validation_pulls.png") + glob.glob("docs/figures/compare_*.png") + glob.glob("docs/figures/benchmark_colab_*.png")):
    print(f); display(Image(f, width=900))

In [ ]:
!zip -qr /content/optphot_colab_results.zip docs/figures results -x "results/compare/*/*.npy"
from google.colab import files
files.download("/content/optphot_colab_results.zip")